# Backfill
* Start by running the stock_df function on Reliance to get the list of all trading dates between 01/01/2008 and 03/11/2026. We then fix time zone issues and save the entire catalogue of dates. This set of dates will be used to get all trading day data records and will be useful to remember what days the Indian stock market was open.
* Important to remember that using bhavcopy to fetch info for 15 June 2021 for instance will be have a time attached given by 14 June 2021 at 1830 which is the UTC time corresponding to midnight in India.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Users/manishkumar/quant')

import os
from datetime import date
from jugaad_data.nse import bhavcopy_save, stock_df
import pandas as pd
from src.download import backfill

RAW = '/Users/manishkumar/quant/data/raw'
PROCESSED = '/Users/manishkumar/quant/data/processed'

In [2]:
reliance_dates = stock_df('RELIANCE', from_date = date(2008,1,1), to_date = date.today())
dates = reliance_dates['DATE'].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata').dt.date
trading_days = sorted(set(dates))
print(len(trading_days), trading_days[0], trading_days[-1])

4666 2008-01-01 2026-11-03


/opt/homebrew/Caskroom/miniforge/base/envs/quant_jvan/lib/python3.10/site-packages/jugaad_data/util.py:40: UserWarning: no explicit representation of timezones available for np.datetime64
  return np.datetime64(dt)


In [3]:
pd.Series(trading_days, name='date').to_csv('/Users/manishkumar/quant/data/processed/trading_days.csv', index=False)

* Having gotten all the trading dates, we will run the backfill in blocks. Start with a block of 5 years from 2008 to 2012.

In [4]:
failed_all = backfill(trading_days, RAW)

pd.DataFrame(failed_all, columns=['date', 'error', 'message']).to_csv('/Users/manishkumar/quant/data/processed/failed_all.csv', index=False)
len(failed_all)

2008-07-28 HTTPError 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=28-Jul-2008&type=equities&mode=single
2008-10-21 HTTPError 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=21-Oct-2008&type=equities&mode=single
2010-05-03 HTTPError 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=03-May-2010&type=equities&mode=single
2010-08-27 HTTPError 404 Client Error: Not Found for

173

In [8]:
didntwork = pd.read_csv('/Users/manishkumar/quant/data/processed/failed_all.csv', parse_dates = ['date'])

In [10]:
retry_all = backfill(didntwork['date'].dt.date.tolist(), RAW)

pd.DataFrame(retry_all, columns=['date', 'error', 'message']).to_csv(
    '/Users/manishkumar/quant/data/processed/failed_retry1.csv', index=False)
len(retry_all)

2013-05-24 HTTPError 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=24-May-2013&type=equities&mode=single
2016-04-22 HTTPError 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=22-Apr-2016&type=equities&mode=single
2016-09-23 HTTPError 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=23-Sep-2016&type=equities&mode=single
2017-05-29 HTTPError 404 Client Error: Not Found for

11

In [17]:
dw_again = pd.read_csv('/Users/manishkumar/quant/data/processed/failed_retry1.csv', parse_dates = ['date'])

In [12]:
retry2 = backfill([d for d, _, _ in retry_all], RAW)
pd.DataFrame(retry2, columns=['date', 'error', 'message']).to_csv('/Users/manishkumar/quant/data/processed/failed_retry2.csv', index=False)
len(retry2)

0

* Check here that the number of files in the data folder match up with the number of trading days and hence our collection is complete.

In [13]:
n_files = len([f for f in os.listdir(RAW) if f.endswith('.csv')])
print(n_files, len(trading_days))

4666 4666
